In [91]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/simbarashemutyambizi/lthc-data/aihw-phe-320-age-specific-percentages-chronic-conditions-cald-2021.xlsx


In [92]:
# Load a specific worksheet by its name
df = pd.read_excel('/kaggle/input/datasets/simbarashemutyambizi/lthc-data/aihw-phe-320-age-specific-percentages-chronic-conditions-cald-2021.xlsx', sheet_name='Table S10',skiprows=3)

# Alternative: Load by sheet index (0 is the first sheet, 1 is the second, etc.)
# df = pd.read_excel('your_file.xlsx', sheet_name=1)
df.head()

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s)
0,"Oceania and Antarctica, nfd",0–10 years,00–44,Persons,Arthritis,n.p,n.p,n.p
1,"Oceania and Antarctica, nfd",0–10 years,00–44,Male,Arthritis,n.p,n.p,n.p
2,"Oceania and Antarctica, nfd",0–10 years,00–44,Female,Arthritis,n.p,n.p,n.p
3,"Oceania and Antarctica, nfd",0–10 years,45–64,Persons,Arthritis,n.p,n.p,n.p
4,"Oceania and Antarctica, nfd",0–10 years,45–64,Male,Arthritis,n.p,n.p,n.p


In [93]:
# Load a specific worksheet by its name
df_two = pd.read_excel('/kaggle/input/datasets/simbarashemutyambizi/lthc-data/aihw-phe-320-age-specific-percentages-chronic-conditions-cald-2021.xlsx', sheet_name='Table S13',skiprows=3)

# Alternative: Load by sheet index (0 is the first sheet, 1 is the second, etc.)
# df = pd.read_excel('your_file.xlsx', sheet_name=1)
df_two.head()

,Language used at home,Proficiency in spoken English,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s)
0,"Northern European Languages, nfd",Very well or well,00–44,Persons,Arthritis,n.p,n.p,n.p
1,"Northern European Languages, nfd",Very well or well,00–44,Male,Arthritis,n.p,n.p,n.p
2,"Northern European Languages, nfd",Very well or well,00–44,Female,Arthritis,n.p,n.p,n.p
3,"Northern European Languages, nfd",Very well or well,45–64,Persons,Arthritis,n.p,n.p,n.p
4,"Northern European Languages, nfd",Very well or well,45–64,Male,Arthritis,n.p,n.p,n.p


In [94]:
print(f"The size of table 10 is {len(df)}")
print(f"The size of table 13 is {len(df_two)}")

The size of table 10 is 63180
The size of table 13 is 57798


Next we are going to concat the data, then replace the missing values that result from the concat function using Gemini

In [95]:
concat_df=pd.concat([df,df_two])

In [96]:
concat_df

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English
0,"Oceania and Antarctica, nfd",0–10 years,00–44,Persons,Arthritis,n.p,n.p,n.p,NaN,NaN
1,"Oceania and Antarctica, nfd",0–10 years,00–44,Male,Arthritis,n.p,n.p,n.p,NaN,NaN
2,"Oceania and Antarctica, nfd",0–10 years,00–44,Female,Arthritis,n.p,n.p,n.p,NaN,NaN
3,"Oceania and Antarctica, nfd",0–10 years,45–64,Persons,Arthritis,n.p,n.p,n.p,NaN,NaN
4,"Oceania and Antarctica, nfd",0–10 years,45–64,Male,Arthritis,n.p,n.p,n.p,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
57793,NaN,NaN,45–64,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57794,NaN,NaN,45–64,Female,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57795,NaN,NaN,65 and over,Persons,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57796,NaN,NaN,65 and over,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all


In [97]:
language_to_country_map = {
    'Northern European Languages, nfd': 'Northern Europe, nfd',
    'Celtic, nfd': 'Western Europe, nfd',
    'Gaelic (Scotland)': 'Scotland',
    'Irish': 'Ireland',
    'Welsh': 'Wales',
    'Celtic, nec': 'Western Europe, nfd',
    'German': 'Germany',
    'Letzeburgish': 'Luxembourg',
    'Yiddish': 'Eastern Europe, nfd',
    'Dutch': 'Netherlands',
    'Frisian': 'Netherlands',
    'Afrikaans': 'South Africa',
    'Scandinavian, nfd': 'Northern Europe, nfd',
    'Danish': 'Denmark',
    'Icelandic': 'Iceland',
    'Norwegian': 'Norway',
    'Swedish': 'Sweden',
    'Scandinavian, nec': 'Northern Europe, nfd',
    'Estonian': 'Estonia',
    'Finnish': 'Finland',
    'Finnish and Related Languages, nec': 'Northern Europe, nfd',
    'French': 'France',
    'Greek': 'Greece',
    'Iberian Romance, nfd': 'Spain',
    'Catalan': 'Andorra',
    'Portuguese': 'Portugal',
    'Spanish': 'Spain',
    'Iberian Romance, nec': 'Spain',
    'Italian': 'Italy',
    'Maltese': 'Malta',
    'Basque': 'Spain',
    'Latin': 'Holy See',
    'Other Southern European Languages, nec': 'Southern and Eastern Europe, nfd',
    'Eastern European Languages, nfd': 'Eastern Europe, nfd',
    'Baltic, nfd': 'Eastern Europe, nfd',
    'Latvian': 'Latvia',
    'Lithuanian': 'Lithuania',
    'Hungarian': 'Hungary',
    'East Slavic, nfd': 'Eastern Europe, nfd',
    'Belorussian': 'Belarus',
    'Russian': 'Russian Federation',
    'Ukrainian': 'Ukraine',
    'South Slavic, nfd': 'South Eastern Europe, nfd',
    'Bosnian': 'Bosnia and Herzegovina',
    'Bulgarian': 'Bulgaria',
    'Croatian': 'Croatia',
    'Macedonian': 'North Macedonia',
    'Serbian': 'Serbia',
    'Slovene': 'Slovenia',
    'Serbo-Croatian/Yugoslavian, so described': 'South Eastern Europe, nfd',
    'Czech': 'Czechia',
    'Polish': 'Poland',
    'Slovak': 'Slovakia',
    'Czechoslovakian, so described': 'Czechia',
    'Albanian': 'Albania',
    'Aromunian (Macedo-Romanian)': 'North Macedonia',
    'Romanian': 'Romania',
    'Romany': 'Eastern Europe, nfd',
    'Other Eastern European Languages, nec': 'Eastern Europe, nfd',
    'Iranic, nfd': 'Middle East, nfd',
    'Kurdish': 'Iraq',
    'Pashto': 'Afghanistan',
    'Balochi': 'Pakistan',
    'Dari': 'Afghanistan',
    'Persian (excluding Dari)': 'Iran',
    'Hazaraghi': 'Afghanistan',
    'Iranic, nec': 'Middle East, nfd',
    'Middle Eastern Semitic Languages, nfd': 'Middle East, nfd',
    'Arabic': 'Saudi Arabia',
    'Hebrew': 'Israel',
    'Assyrian Neo-Aramaic': 'Iraq',
    'Chaldean Neo-Aramaic': 'Iraq',
    'Mandaean (Mandaic)': 'Iraq',
    'Middle Eastern Semitic Languages, nec': 'Middle East, nfd',
    'Turkic, nfd': 'Central Asia, nfd',
    'Turkish': 'Turkey',
    'Azeri': 'Azerbaijan',
    'Tatar': 'Russian Federation',
    'Turkmen': 'Turkmenistan',
    'Uygur': 'China (excludes SARs and Taiwan)',
    'Uzbek': 'Uzbekistan',
    'Turkic, nec': 'Central Asia, nfd',
    'Armenian': 'Armenia',
    'Georgian': 'Georgia',
    'Other Southwest and Central Asian Languages, n': 'Central Asia, nfd',
    'Southern Asian Languages, nfd': 'Southern Asia, nfd',
    'Kannada': 'India',
    'Malayalam': 'India',
    'Tamil': 'India',
    'Telugu': 'India',
    'Tulu': 'India',
    'Dravidian, nec': 'India',
    'Indo-Aryan, nfd': 'Southern Asia, nfd',
    'Bengali': 'Bangladesh',
    'Gujarati': 'India',
    'Hindi': 'India',
    'Konkani': 'India',
    'Marathi': 'India',
    'Nepali': 'Nepal',
    'Punjabi': 'India',
    'Sindhi': 'Pakistan',
    'Sinhalese': 'Sri Lanka',
    'Urdu': 'Pakistan',
    'Assamese': 'India',
    'Dhivehi': 'Maldives',
    'Kashmiri': 'India',
    'Oriya': 'India',
    'Fijian Hindustani': 'Fiji',
    'Indo-Aryan, nec': 'Southern Asia, nfd',
    'Other Southern Asian Languages': 'Southern Asia, nfd',
    'Southeast Asian Languages, nfd': 'South-East Asia, nfd',
    'Burmese and Related Languages, nfd': 'Myanmar',
    'Burmese': 'Myanmar',
    'Chin Haka': 'Myanmar',
    'Karen': 'Myanmar',
    'Rohingya': 'Myanmar',
    'Zomi': 'Myanmar',
    'Burmese and Related Languages, nec': 'Myanmar',
    'Hmong': 'Laos',
    'Khmer': 'Cambodia',
    'Vietnamese': 'Vietnam',
    'Mon': 'Myanmar',
    'Mon-Khmer, nec': 'Mainland South-East Asia, nfd',
    'Lao': 'Laos',
    'Thai': 'Thailand',
    'Tai, nec': 'Mainland South-East Asia, nfd',
    'Southeast Asian Austronesian Languages, nfd': 'Maritime South-East Asia, nfd',
    'Bisaya': 'Philippines',
    'Cebuano': 'Philippines',
    'IIokano': 'Philippines',
    'Indonesian': 'Indonesia',
    'Malay': 'Malaysia',
    'Tetum': 'Timor-Leste',
    'Timorese': 'Timor-Leste',
    'Tagalog': 'Philippines',
    'Filipino': 'Philippines',
    'Acehnese': 'Indonesia',
    'Balinese': 'Indonesia',
    'Bikol': 'Philippines',
    'Iban': 'Malaysia',
    'Ilonggo (Hiligaynon)': 'Philippines',
    'Javanese': 'Indonesia',
    'Pampangan': 'Philippines',
    'Southeast Asian Austronesian Languages, nec': 'Maritime South-East Asia, nfd',
    'Other Southeast Asian Languages': 'South-East Asia, nfd',
    'Eastern Asian Languages, nfd': 'Chinese Asia (includes Mongolia), nfd',
    'Chinese, nfd': 'China (excludes SARs and Taiwan)',
    'Cantonese': 'Hong Kong (SAR of China)',
    'Hakka': 'China (excludes SARs and Taiwan)',
    'Mandarin': 'China (excludes SARs and Taiwan)',
    'Wu': 'China (excludes SARs and Taiwan)',
    'Min Nan': 'Taiwan',
    'Chinese, nec': 'China (excludes SARs and Taiwan)',
    'Japanese': 'Japan',
    'Korean': 'Korea, Republic of (South)',
    'Tibetan': 'China (excludes SARs and Taiwan)',
    'Mongolian': 'Mongolia',
    'Other Eastern Asian Languages, nec': 'Chinese Asia (includes Mongolia), nfd',
    '.ATSI lang': 'Oceania and Antarctica, nfd',
    'Other Languages, nfd': 'Americas, nfd',
    'Non-verbal, so described': 'Americas, nfd',
    'Swiss, so described': 'Switzerland',
    'Cypriot, so described': 'Cyprus',
    'Creole, nfd': 'Caribbean, nfd',
    'French Creole, nfd': 'Haiti',
    'Spanish Creole, nfd': 'Dominican Republic',
    'Portuguese Creole, nfd': 'Cabo Verde',
    'Pidgin, nfd': 'Oceania and Antarctica, nfd',
    'American Languages': 'United States of America',
    'African Languages, nfd': 'Sub-Saharan Africa, nfd',
    'Acholi': 'Uganda',
    'Akan': 'Ghana',
    'Mauritian Creole': 'Mauritius',
    'Oromo': 'Ethiopia',
    'Shona': 'Zimbabwe',
    'Somali': 'Somalia',
    'Swahili': 'Tanzania',
    'Yoruba': 'Nigeria',
    'Zulu': 'South Africa',
    'Amharic': 'Ethiopia',
    'Bemba': 'Zambia',
    'Dinka': 'South Sudan',
    'Ewe': 'Ghana',
    'Ga': 'Ghana',
    'Harari': 'Ethiopia',
    'Hausa': 'Nigeria',
    'Igbo': 'Nigeria',
    'Kikuyu': 'Kenya',
    'Krio': 'Sierra Leone',
    'Luganda': 'Uganda',
    'Luo': 'Kenya',
    'Ndebele': 'Zimbabwe',
    'Nuer': 'South Sudan',
    'Nyanja (Chichewa)': 'Malawi',
    'Shilluk': 'South Sudan',
    'Tigre': 'Eritrea',
    'Tigrinya': 'Eritrea',
    'Tswana': 'Botswana',
    'Xhosa': 'South Africa',
    'Seychelles Creole': 'Seychelles',
    'Anuak': 'South Sudan',
    'Bari': 'South Sudan',
    'Bassa': 'Liberia',
    'Dan (Gio-Dan)': 'Liberia',
    'Fulfulde': 'Nigeria',
    'Kinyarwanda (Rwanda)': 'Rwanda',
    'Kirundi (Rundi)': 'Burundi',
    'Kpelle': 'Liberia',
    'Krahn': 'Liberia',
    'Liberian (Liberian English)': 'Liberia',
    'Loma (Lorma)': 'Liberia',
    'Madi': 'Uganda',
    'Mandinka': 'Gambia',
    'Mann': 'Liberia',
    'Moro (Nuba Moro)': 'Sudan',
    'Themne': 'Sierra Leone',
    'Lingala': 'Congo, Democratic Republic of',
    'African Languages, nec': 'Sub-Saharan Africa, nfd',
    'Pacific Austronesian Languages, nfd': 'Polynesia (excludes Hawaii), nec',
    'Fijian': 'Fiji',
    'Gilbertese': 'Kiribati',
    'Maori (Cook Island)': 'Cook Islands',
    'Maori (New Zealand)': 'New Zealand',
    'Nauruan': 'Nauru',
    'Niue': 'Niue',
    'Samoan': 'Samoa',
    'Tongan': 'Tonga',
    'Rotuman': 'Fiji',
    'Tokelauan': 'Tokelau',
    'Tuvaluan': 'Tuvalu',
    'Yapese': 'Micronesia, Federated States of',
    'Pacific Austronesian Languages, nec': 'Polynesia (excludes Hawaii), nec',
    'Oceanian Pidgins and Creoles, nfd': 'Oceania and Antarctica, nfd',
    'Bislama': 'Vanuatu',
    "Norf'k-Pitcairn": 'Pitcairn Islands',
    'Solomon Islands Pijin': 'Solomon Islands',
    'Oceanian Pidgins and Creoles, nec': 'Oceania and Antarctica, nfd',
    'Papua New Guinea Languages, nfd': 'Papua New Guinea',
    'Kiwai': 'Papua New Guinea',
    'Motu (HiriMotu)': 'Papua New Guinea',
    'Tok Pisin (Neomelanesian)': 'Papua New Guinea',
    'Papua New Guinea Languages, nec': 'Papua New Guinea',
    'Invented Languages': 'Western Europe, nfd',
    'Sign Languages, nfd': 'Americas, nfd',
    'Auslan': 'Oceania and Antarctica, nfd',
    'Key Word Sign Australia': 'Oceania and Antarctica, nfd',
    'Sign Languages, nec': 'Americas, nfd'
}

In [98]:
country_to_languages = {
    'Melanesia, nfd': 'Pacific Austronesian Languages, nfd',
    'New Caledonia': 'French',
    'Micronesia, nfd': 'Pacific Austronesian Languages, nfd',
    'Guam': 'Tagalog',
    'Marshall Islands': 'Pacific Austronesian Languages, nfd',
    'Northern Mariana Islands': 'Tagalog',
    'Palau': 'Pacific Austronesian Languages, nfd',
    'French Polynesia': 'French',
    'Samoa, American': 'Samoan',
    'Wallis and Futuna': 'French',
    'Antarctica, nfd': 'Other Languages, nfd',
    'Chilean Antarctic Territory': 'Spanish',
    'United Kingdom, Channel Islands and Isle of Ma': 'Northern European Languages, nfd',
    'England': 'Northern European Languages, nfd',
    'Isle of Man': 'Celtic, nfd',
    'Northern Ireland': 'Irish',
    'Guernsey': 'French',
    'Jersey': 'French',
    'Austria': 'German',
    'Belgium': 'Dutch',
    'Liechtenstein': 'German',
    'Monaco': 'French',
    'Faroe Islands': 'Danish',
    'Greenland': 'Danish',
    'Aland Islands': 'Swedish',
    'Gibraltar': 'Spanish',
    'San Marino': 'Italian',
    'Moldova': 'Romanian',
    'Montenegro': 'Serbian',
    'Kosovo': 'Albanian',
    'North Africa and the Middle East, nfd': 'Arabic',
    'North Africa, nfd': 'Arabic',
    'Algeria': 'Arabic',
    'Egypt': 'Arabic',
    'Libya': 'Arabic',
    'Morocco': 'Arabic',
    'Tunisia': 'Arabic',
    'Western Sahara': 'Arabic',
    'Bahrain': 'Arabic',
    'Gaza Strip and West Bank': 'Arabic',
    'Jordan': 'Arabic',
    'Kuwait': 'Arabic',
    'Lebanon': 'Arabic',
    'Oman': 'Arabic',
    'Qatar': 'Arabic',
    'Syria': 'Arabic',
    'United Arab Emirates': 'Arabic',
    'Yemen': 'Arabic',
    'Brunei Darussalam': 'Malay',
    'Singapore': 'Mandarin',
    'Macau (SAR of China)': 'Cantonese',
    "Korea, Democratic People's Republic of (North)": 'Korean',
    'Southern and Central Asia, nfd': 'Southern Asian Languages, nfd',
    'Bhutan': 'Tibetan',
    'Kazakhstan': 'Russian',
    'Kyrgyzstan': 'Russian',
    'Tajikistan': 'Persian (excluding Dari)',
    'Northern America, nfd': 'Northern European Languages, nfd',
    'Bermuda': 'Creole, nfd',
    'Canada': 'French',
    'South America, nfd': 'Spanish',
    'Argentina': 'Spanish',
    'Bolivia': 'Spanish',
    'Brazil': 'Portuguese',
    'Chile': 'Spanish',
    'Colombia': 'Spanish',
    'Ecuador': 'Spanish',
    'Falkland Islands': 'Spanish',
    'French Guiana': 'French',
    'Guyana': 'Creole, nfd',
    'Paraguay': 'Spanish',
    'Peru': 'Spanish',
    'Suriname': 'Dutch',
    'Uruguay': 'Spanish',
    'Venezuela': 'Spanish',
    'Central America, nfd': 'Spanish',
    'Belize': 'Spanish',
    'Costa Rica': 'Spanish',
    'El Salvador': 'Spanish',
    'Guatemala': 'Spanish',
    'Honduras': 'Spanish',
    'Mexico': 'Spanish',
    'Nicaragua': 'Spanish',
    'Panama': 'Spanish',
    'Anguilla': 'Creole, nfd',
    'Antigua and Barbuda': 'Creole, nfd',
    'Aruba': 'Dutch',
    'Bahamas': 'Creole, nfd',
    'Barbados': 'Creole, nfd',
    'Cayman Islands': 'Creole, nfd',
    'Cuba': 'Spanish',
    'Dominica': 'French Creole, nfd',
    'Grenada': 'Creole, nfd',
    'Guadeloupe': 'French Creole, nfd',
    'Jamaica': 'Creole, nfd',
    'Martinique': 'French Creole, nfd',
    'Montserrat': 'Creole, nfd',
    'Puerto Rico': 'Spanish',
    'St Kitts and Nevis': 'Creole, nfd',
    'St Lucia': 'French Creole, nfd',
    'St Vincent and the Grenadines': 'Creole, nfd',
    'Trinidad and Tobago': 'Creole, nfd',
    'Turks and Caicos Islands': 'Creole, nfd',
    'Virgin Islands, British': 'Creole, nfd',
    'Virgin Islands, United States': 'Creole, nfd',
    'St Martin (French part)': 'French',
    'Bonaire, Sint Eustatius and Saba': 'Dutch',
    'Curacao': 'Dutch',
    'Sint Maarten (Dutch part)': 'Dutch',
    'Central and West Africa, nfd': 'African Languages, nfd',
    'Benin': 'French',
    'Burkina Faso': 'French',
    'Cameroon': 'French',
    'Central African Republic': 'French',
    'Chad': 'Arabic',
    'Congo, Republic of': 'French',
    "Cote d'Ivoire": 'French',
    'Equatorial Guinea': 'Spanish',
    'Gabon': 'French',
    'Guinea': 'French',
    'Guinea-Bissau': 'Portuguese',
    'Mali': 'French',
    'Mauritania': 'Arabic',
    'Niger': 'French',
    'Sao Tome and Principe': 'Portuguese',
    'Senegal': 'French',
    'Togo': 'French',
    'Southern and East Africa, nfd': 'African Languages, nfd',
    'Angola': 'Portuguese',
    'Comoros': 'Arabic',
    'Djibouti': 'Arabic',
    'Lesotho': 'African Languages, nfd',
    'Madagascar': 'Pacific Austronesian Languages, nfd',
    'Mayotte': 'French',
    'Mozambique': 'Portuguese',
    'Namibia': 'Afrikaans',
    'Reunion': 'French Creole, nfd',
    'St Helena': 'Northern European Languages, nfd',
    'Eswatini': 'Zulu'
}

We took all the countries from the dataset and all the languages as well and mapped them to each other. Some countries have more than one language. We're investigating that next

In [99]:
languages_to_country_count=pd.Series(list(language_to_country_map.values())).value_counts().reset_index()

In [100]:
languages_to_country_count.loc[languages_to_country_count['count']>1]

,index,count
0,India,14
1,Myanmar,8
2,Philippines,8
3,China (excludes SARs and Taiwan),7
4,Liberia,7
5,"Oceania and Antarctica, nfd",6
6,"Eastern Europe, nfd",6
7,South Sudan,5
8,Papua New Guinea,5
9,"Northern Europe, nfd",4


In [101]:
languages_to_country_count

,index,count
0,India,14
1,Myanmar,8
2,Philippines,8
3,China (excludes SARs and Taiwan),7
4,Liberia,7
...,...,...
125,Tokelau,1
126,"Micronesia, Federated States of",1
127,Vanuatu,1
128,Pitcairn Islands,1


In [102]:
df.columns

Index(['Country of birth of person', 'Years spent in Australia', 'Age group',
       'Sex', 'Long-term health condition (LTHC)',
       'Number of people reporting LTHC(s)', 'Population',
       'Age-specific percentage of population reporting LTHC(s)'],
      dtype='object')

In [103]:
combined_df=pd.concat([df,df_two])

In [104]:
def common_language_finder(language_to_country_map,df,language_to_country_count):
    
    country_list=language_to_country_count["index"].to_list()
    country_to_common_language_dic={}
    for n in country_list:
        language_of_country=[key for key,value in language_to_country_map.items() if value == n]
        ##retrieving the languages count
        dummy_var=0
        for z in language_of_country:
            if df["Language used at home"].loc[df["Language used at home"]==z].count() >dummy_var:
                language_holder=z
                dummy_var=df["Language used at home"].loc[df["Language used at home"]==z].count()
            else:
                continue
           
        country_to_common_language_dic[n]=language_holder    
    
    return country_to_common_language_dic    


country_to_common_language_dic=common_language_finder(language_to_country_map,df_two,languages_to_country_count)      
    
    
    

In [105]:
combined_df["Language used at home"]=combined_df["Language used at home"].fillna(combined_df["Country of birth of person"].map(country_to_common_language_dic))
combined_df

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English
0,"Oceania and Antarctica, nfd",0–10 years,00–44,Persons,Arthritis,n.p,n.p,n.p,.ATSI lang,NaN
1,"Oceania and Antarctica, nfd",0–10 years,00–44,Male,Arthritis,n.p,n.p,n.p,.ATSI lang,NaN
2,"Oceania and Antarctica, nfd",0–10 years,00–44,Female,Arthritis,n.p,n.p,n.p,.ATSI lang,NaN
3,"Oceania and Antarctica, nfd",0–10 years,45–64,Persons,Arthritis,n.p,n.p,n.p,.ATSI lang,NaN
4,"Oceania and Antarctica, nfd",0–10 years,45–64,Male,Arthritis,n.p,n.p,n.p,.ATSI lang,NaN
...,...,...,...,...,...,...,...,...,...,...
57793,NaN,NaN,45–64,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57794,NaN,NaN,45–64,Female,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57795,NaN,NaN,65 and over,Persons,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57796,NaN,NaN,65 and over,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all


In [106]:
combined_df["Language used at home"]=combined_df["Language used at home"].fillna(combined_df["Country of birth of person"].map(country_to_languages))

In [107]:
combined_df.isnull().sum()

Country of birth of person                                 57798
Years spent in Australia                                   57798
Age group                                                      0
Sex                                                            0
Long-term health condition (LTHC)                              0
Number of people reporting LTHC(s)                             0
Population                                                     0
Age-specific percentage of population reporting LTHC(s)        0
Language used at home                                        234
Proficiency in spoken English                              63180
dtype: int64

## Now we will proceed to fix the null values fro proficieny in spoken english

In [108]:
language_to_country_map["Shona"]

'Zimbabwe'

In [109]:
mask = (concat_df["Language used at home"] == "Shona") & concat_df["Country of birth of person"].isna()

concat_df.loc[mask, "Country of birth of person"] = language_to_country_map["Shona"]

In [110]:
languages=[keys for keys,values in language_to_country_map.items()]

In [111]:
languages

['Northern European Languages, nfd',
 'Celtic, nfd',
 'Gaelic (Scotland)',
 'Irish',
 'Welsh',
 'Celtic, nec',
 'German',
 'Letzeburgish',
 'Yiddish',
 'Dutch',
 'Frisian',
 'Afrikaans',
 'Scandinavian, nfd',
 'Danish',
 'Icelandic',
 'Norwegian',
 'Swedish',
 'Scandinavian, nec',
 'Estonian',
 'Finnish',
 'Finnish and Related Languages, nec',
 'French',
 'Greek',
 'Iberian Romance, nfd',
 'Catalan',
 'Portuguese',
 'Spanish',
 'Iberian Romance, nec',
 'Italian',
 'Maltese',
 'Basque',
 'Latin',
 'Other Southern European Languages, nec',
 'Eastern European Languages, nfd',
 'Baltic, nfd',
 'Latvian',
 'Lithuanian',
 'Hungarian',
 'East Slavic, nfd',
 'Belorussian',
 'Russian',
 'Ukrainian',
 'South Slavic, nfd',
 'Bosnian',
 'Bulgarian',
 'Croatian',
 'Macedonian',
 'Serbian',
 'Slovene',
 'Serbo-Croatian/Yugoslavian, so described',
 'Czech',
 'Polish',
 'Slovak',
 'Czechoslovakian, so described',
 'Albanian',
 'Aromunian (Macedo-Romanian)',
 'Romanian',
 'Romany',
 'Other Eastern Euro

In [112]:
def country_based_on_language(df,languages,language_to_country_map):
    languages_without_country=[]
    for n in languages:
        if language_to_country_map[n] != None:
            mask = (df["Language used at home"] == n) & df["Country of birth of person"].isna()
    
            df.loc[mask, "Country of birth of person"] = language_to_country_map[n]
        else:
            languages_without_country.append(n)
    return df,languages_without_country       
            
combined_df,languages_without_country=country_based_on_language(combined_df,languages,language_to_country_map)            
        

In [113]:
combined_df.isnull().sum()

Country of birth of person                                     0
Years spent in Australia                                   57798
Age group                                                      0
Sex                                                            0
Long-term health condition (LTHC)                              0
Number of people reporting LTHC(s)                             0
Population                                                     0
Age-specific percentage of population reporting LTHC(s)        0
Language used at home                                        234
Proficiency in spoken English                              63180
dtype: int64

## Filling null values of the remaining columns

In [114]:
'''
def filtered_df(df): 
    result_df=df.groupby([
            'Country of birth of person', 
            'Age group',
            'Sex', 
            'Long-term health condition (LTHC)',
            'Language used at home'
        ])["Years spent in Australia"].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else None).reset_index(name="Mode years spent in Australia")
    filtered_df=result_df.loc[result_df["Mode years spent in Australia"].notna()]
    



    
    for n in range(0,len(filtered_df)):
        cols = ["Country of birth of person", "Age group", "Sex", "Long-term health condition (LTHC)", "Language used at home"]
        cc=filtered_df[cols].iloc[n]
        
        mask = (df[cols] == cc).all(axis=1)
        df.loc[mask].fillna(filtered_df["Mode years spent in Australia"].iloc[n])
        
    return df    
    '''

'\ndef filtered_df(df): \n    result_df=df.groupby([\n            \'Country of birth of person\', \n            \'Age group\',\n            \'Sex\', \n            \'Long-term health condition (LTHC)\',\n            \'Language used at home\'\n        ])["Years spent in Australia"].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else None).reset_index(name="Mode years spent in Australia")\n    filtered_df=result_df.loc[result_df["Mode years spent in Australia"].notna()]\n    \n\n\n\n    \n    for n in range(0,len(filtered_df)):\n        cols = ["Country of birth of person", "Age group", "Sex", "Long-term health condition (LTHC)", "Language used at home"]\n        cc=filtered_df[cols].iloc[n]\n        \n        mask = (df[cols] == cc).all(axis=1)\n        df.loc[mask].fillna(filtered_df["Mode years spent in Australia"].iloc[n])\n        \n    return df    \n    '

The code above take too long to process, so we will just impute null values with mode

In [115]:
combined_df["Years spent in Australia"].fillna("Not specified",inplace=True)
combined_df["Proficiency in spoken English"].fillna("Not specified",inplace=True)

/tmp/ipykernel_111/3207283105.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined_df["Years spent in Australia"].fillna("Not specified",inplace=True)
/tmp/ipykernel_111/3207283105.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(v

In [116]:
combined_df["Language used at home"].fillna("Not specified",inplace=True)

/tmp/ipykernel_111/3636230363.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined_df["Language used at home"].fillna("Not specified",inplace=True)


In [117]:
combined_df

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English
0,"Oceania and Antarctica, nfd",0–10 years,00–44,Persons,Arthritis,n.p,n.p,n.p,.ATSI lang,Not specified
1,"Oceania and Antarctica, nfd",0–10 years,00–44,Male,Arthritis,n.p,n.p,n.p,.ATSI lang,Not specified
2,"Oceania and Antarctica, nfd",0–10 years,00–44,Female,Arthritis,n.p,n.p,n.p,.ATSI lang,Not specified
3,"Oceania and Antarctica, nfd",0–10 years,45–64,Persons,Arthritis,n.p,n.p,n.p,.ATSI lang,Not specified
4,"Oceania and Antarctica, nfd",0–10 years,45–64,Male,Arthritis,n.p,n.p,n.p,.ATSI lang,Not specified
...,...,...,...,...,...,...,...,...,...,...
57793,"Americas, nfd",Not specified,45–64,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57794,"Americas, nfd",Not specified,45–64,Female,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57795,"Americas, nfd",Not specified,65 and over,Persons,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all
57796,"Americas, nfd",Not specified,65 and over,Male,One or more long-term health condition(s),n.p,n.p,n.p,"Sign Languages, nec",Not well or Not at all


In [118]:
cc=combined_df.loc[combined_df["Age-specific percentage of population reporting LTHC(s)"]!="n.p"]
#cc=cc.loc[cc["Country of birth of person"]=="Oceania and Antarctica, nfd"]["Long-term health condition (LTHC)"].value_counts()
cc

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English
18,New Zealand,0–10 years,00–44,Persons,Arthritis,1005,100403,1.000966,Maori (New Zealand),Not specified
19,New Zealand,0–10 years,00–44,Male,Arthritis,453,50915,0.889718,Maori (New Zealand),Not specified
20,New Zealand,0–10 years,00–44,Female,Arthritis,552,49488,1.115422,Maori (New Zealand),Not specified
21,New Zealand,0–10 years,45–64,Persons,Arthritis,1323,17280,7.65625,Maori (New Zealand),Not specified
22,New Zealand,0–10 years,45–64,Male,Arthritis,427,8301,5.143959,Maori (New Zealand),Not specified
...,...,...,...,...,...,...,...,...,...,...
57757,"Oceania and Antarctica, nfd",Not specified,45–64,Male,One or more long-term health condition(s),301,483,62.318841,Auslan,Not well or Not at all
57758,"Oceania and Antarctica, nfd",Not specified,45–64,Female,One or more long-term health condition(s),295,485,60.824742,Auslan,Not well or Not at all
57759,"Oceania and Antarctica, nfd",Not specified,65 and over,Persons,One or more long-term health condition(s),341,498,68.473896,Auslan,Not well or Not at all
57760,"Oceania and Antarctica, nfd",Not specified,65 and over,Male,One or more long-term health condition(s),186,280,66.428571,Auslan,Not well or Not at all


In [119]:
combined_df.duplicated().sum()

np.int64(0)

In [120]:
combined_df.isnull().sum()

Country of birth of person                                 0
Years spent in Australia                                   0
Age group                                                  0
Sex                                                        0
Long-term health condition (LTHC)                          0
Number of people reporting LTHC(s)                         0
Population                                                 0
Age-specific percentage of population reporting LTHC(s)    0
Language used at home                                      0
Proficiency in spoken English                              0
dtype: int64

In [121]:
combined_df.dtypes

Country of birth of person                                 object
Years spent in Australia                                   object
Age group                                                  object
Sex                                                        object
Long-term health condition (LTHC)                          object
Number of people reporting LTHC(s)                         object
Population                                                 object
Age-specific percentage of population reporting LTHC(s)    object
Language used at home                                      object
Proficiency in spoken English                              object
dtype: object

In [122]:
# List of columns to clean
cols_to_clean = [
    "Population", 
    "Age-specific percentage of population reporting LTHC(s)", 
    "Number of people reporting LTHC(s)"
]

# Replace "n.p" with 0.0, then convert the columns to floats
combined_df[cols_to_clean] = combined_df[cols_to_clean].replace("n.p", 0.0).astype(float)

/tmp/ipykernel_111/1188944814.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined_df[cols_to_clean] = combined_df[cols_to_clean].replace("n.p", 0.0).astype(float)


In [123]:
combined_df.dtypes

Country of birth of person                                  object
Years spent in Australia                                    object
Age group                                                   object
Sex                                                         object
Long-term health condition (LTHC)                           object
Number of people reporting LTHC(s)                         float64
Population                                                 float64
Age-specific percentage of population reporting LTHC(s)    float64
Language used at home                                       object
Proficiency in spoken English                               object
dtype: object

In [124]:
combined_df.isnull().sum()

Country of birth of person                                 0
Years spent in Australia                                   0
Age group                                                  0
Sex                                                        0
Long-term health condition (LTHC)                          0
Number of people reporting LTHC(s)                         0
Population                                                 0
Age-specific percentage of population reporting LTHC(s)    0
Language used at home                                      0
Proficiency in spoken English                              0
dtype: int64

In [125]:
combined_df

,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English
0,"Oceania and Antarctica, nfd",0–10 years,00–44,Persons,Arthritis,0.0,0.0,0.0,.ATSI lang,Not specified
1,"Oceania and Antarctica, nfd",0–10 years,00–44,Male,Arthritis,0.0,0.0,0.0,.ATSI lang,Not specified
2,"Oceania and Antarctica, nfd",0–10 years,00–44,Female,Arthritis,0.0,0.0,0.0,.ATSI lang,Not specified
3,"Oceania and Antarctica, nfd",0–10 years,45–64,Persons,Arthritis,0.0,0.0,0.0,.ATSI lang,Not specified
4,"Oceania and Antarctica, nfd",0–10 years,45–64,Male,Arthritis,0.0,0.0,0.0,.ATSI lang,Not specified
...,...,...,...,...,...,...,...,...,...,...
57793,"Americas, nfd",Not specified,45–64,Male,One or more long-term health condition(s),0.0,0.0,0.0,"Sign Languages, nec",Not well or Not at all
57794,"Americas, nfd",Not specified,45–64,Female,One or more long-term health condition(s),0.0,0.0,0.0,"Sign Languages, nec",Not well or Not at all
57795,"Americas, nfd",Not specified,65 and over,Persons,One or more long-term health condition(s),0.0,0.0,0.0,"Sign Languages, nec",Not well or Not at all
57796,"Americas, nfd",Not specified,65 and over,Male,One or more long-term health condition(s),0.0,0.0,0.0,"Sign Languages, nec",Not well or Not at all


In [126]:
country_to_subregion = {
    # Oceania and Antarctica
    'Oceania and Antarctica, nfd': 'Oceania (nfd)',
    'New Zealand': 'New Zealand',
    'Melanesia, nfd': 'Melanesia',
    'New Caledonia': 'Melanesia',
    'Papua New Guinea': 'Melanesia',
    'Solomon Islands': 'Melanesia',
    'Vanuatu': 'Melanesia',
    'Micronesia, nfd': 'Micronesia',
    'Guam': 'Micronesia',
    'Kiribati': 'Micronesia',
    'Marshall Islands': 'Micronesia',
    'Micronesia, Federated States of': 'Micronesia',
    'Nauru': 'Micronesia',
    'Northern Mariana Islands': 'Micronesia',
    'Palau': 'Micronesia',
    'Cook Islands': 'Polynesia',
    'Fiji': 'Polynesia',
    'French Polynesia': 'Polynesia',
    'Niue': 'Polynesia',
    'Samoa': 'Polynesia',
    'Samoa, American': 'Polynesia',
    'Tokelau': 'Polynesia',
    'Tonga': 'Polynesia',
    'Tuvalu': 'Polynesia',
    'Wallis and Futuna': 'Polynesia',
    'Pitcairn Islands': 'Polynesia',
    'Polynesia (excludes Hawaii), nec': 'Polynesia',
    'Antarctica, nfd': 'Antarctica',
    'Chilean Antarctic Territory': 'Antarctica',

    # North-West Europe
    'United Kingdom, Channel Islands and Isle of Ma': 'United Kingdom',
    'England': 'United Kingdom',
    'Isle of Man': 'United Kingdom',
    'Northern Ireland': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'Wales': 'United Kingdom',
    'Guernsey': 'United Kingdom',
    'Jersey': 'United Kingdom',
    'Ireland': 'Ireland',
    'Western Europe, nfd': 'Western Europe',
    'Austria': 'Western Europe',
    'Belgium': 'Western Europe',
    'France': 'Western Europe',
    'Germany': 'Western Europe',
    'Liechtenstein': 'Western Europe',
    'Luxembourg': 'Western Europe',
    'Monaco': 'Western Europe',
    'Netherlands': 'Western Europe',
    'Switzerland': 'Western Europe',
    'Northern Europe, nfd': 'Northern Europe',
    'Denmark': 'Northern Europe',
    'Faroe Islands': 'Northern Europe',
    'Finland': 'Northern Europe',
    'Greenland': 'Northern Europe',
    'Iceland': 'Northern Europe',
    'Norway': 'Northern Europe',
    'Sweden': 'Northern Europe',
    'Aland Islands': 'Northern Europe',

    # Southern and Eastern Europe
    'Southern and Eastern Europe, nfd': 'Southern Europe',
    'Andorra': 'Southern Europe',
    'Gibraltar': 'Southern Europe',
    'Holy See': 'Southern Europe',
    'Italy': 'Southern Europe',
    'Malta': 'Southern Europe',
    'Portugal': 'Southern Europe',
    'San Marino': 'Southern Europe',
    'Spain': 'Southern Europe',
    'South Eastern Europe, nfd': 'South Eastern Europe',
    'Albania': 'South Eastern Europe',
    'Bosnia and Herzegovina': 'South Eastern Europe',
    'Bulgaria': 'South Eastern Europe',
    'Croatia': 'South Eastern Europe',
    'Cyprus': 'South Eastern Europe',
    'North Macedonia': 'South Eastern Europe',
    'Greece': 'South Eastern Europe',
    'Moldova': 'South Eastern Europe',
    'Romania': 'South Eastern Europe',
    'Slovenia': 'South Eastern Europe',
    'Montenegro': 'South Eastern Europe',
    'Serbia': 'South Eastern Europe',
    'Kosovo': 'South Eastern Europe',
    'Eastern Europe, nfd': 'Eastern Europe',
    'Belarus': 'Eastern Europe',
    'Czechia': 'Eastern Europe',
    'Estonia': 'Eastern Europe',
    'Hungary': 'Eastern Europe',
    'Latvia': 'Eastern Europe',
    'Lithuania': 'Eastern Europe',
    'Poland': 'Eastern Europe',
    'Russian Federation': 'Eastern Europe',
    'Slovakia': 'Eastern Europe',
    'Ukraine': 'Eastern Europe',

    # North Africa and the Middle East
    'North Africa and the Middle East, nfd': 'North Africa and Middle East (nfd)',
    'North Africa, nfd': 'North Africa',
    'Algeria': 'North Africa',
    'Egypt': 'North Africa',
    'Libya': 'North Africa',
    'Morocco': 'North Africa',
    'Sudan': 'North Africa',
    'Tunisia': 'North Africa',
    'Western Sahara': 'North Africa',
    'South Sudan': 'North Africa',
    'Middle East, nfd': 'Middle East',
    'Bahrain': 'Middle East',
    'Gaza Strip and West Bank': 'Middle East',
    'Iran': 'Middle East',
    'Iraq': 'Middle East',
    'Israel': 'Middle East',
    'Jordan': 'Middle East',
    'Kuwait': 'Middle East',
    'Lebanon': 'Middle East',
    'Oman': 'Middle East',
    'Qatar': 'Middle East',
    'Saudi Arabia': 'Middle East',
    'Syria': 'Middle East',
    'Turkey': 'Middle East',
    'United Arab Emirates': 'Middle East',
    'Yemen': 'Middle East',

    # South-East Asia
    'South-East Asia, nfd': 'South-East Asia (nfd)',
    'Mainland South-East Asia, nfd': 'Mainland South-East Asia',
    'Myanmar': 'Mainland South-East Asia',
    'Cambodia': 'Mainland South-East Asia',
    'Laos': 'Mainland South-East Asia',
    'Thailand': 'Mainland South-East Asia',
    'Vietnam': 'Mainland South-East Asia',
    'Maritime South-East Asia, nfd': 'Maritime South-East Asia',
    'Brunei Darussalam': 'Maritime South-East Asia',
    'Indonesia': 'Maritime South-East Asia',
    'Malaysia': 'Maritime South-East Asia',
    'Philippines': 'Maritime South-East Asia',
    'Singapore': 'Maritime South-East Asia',
    'Timor-Leste': 'Maritime South-East Asia',

    # North-East Asia
    'Chinese Asia (includes Mongolia), nfd': 'Chinese Asia',
    'China (excludes SARs and Taiwan)': 'Chinese Asia',
    'Hong Kong (SAR of China)': 'Chinese Asia',
    'Macau (SAR of China)': 'Chinese Asia',
    'Mongolia': 'Chinese Asia',
    'Taiwan': 'Chinese Asia',
    'Japan': 'Japan and the Koreas',
    "Korea, Democratic People's Republic of (North)": 'Japan and the Koreas',
    'Korea, Republic of (South)': 'Japan and the Koreas',

    # Southern and Central Asia
    'Southern and Central Asia, nfd': 'Southern and Central Asia (nfd)',
    'Southern Asia, nfd': 'Southern Asia',
    'Bangladesh': 'Southern Asia',
    'Bhutan': 'Southern Asia',
    'India': 'Southern Asia',
    'Maldives': 'Southern Asia',
    'Nepal': 'Southern Asia',
    'Pakistan': 'Southern Asia',
    'Sri Lanka': 'Southern Asia',
    'Central Asia, nfd': 'Central Asia',
    'Afghanistan': 'Central Asia',
    'Armenia': 'Central Asia',
    'Azerbaijan': 'Central Asia',
    'Georgia': 'Central Asia',
    'Kazakhstan': 'Central Asia',
    'Kyrgyzstan': 'Central Asia',
    'Tajikistan': 'Central Asia',
    'Turkmenistan': 'Central Asia',
    'Uzbekistan': 'Central Asia',

    # Americas
    'Americas, nfd': 'Americas (nfd)',
    'Northern America, nfd': 'Northern America',
    'Bermuda': 'Northern America',
    'Canada': 'Northern America',
    'United States of America': 'Northern America',
    'South America, nfd': 'South America',
    'Argentina': 'South America',
    'Bolivia': 'South America',
    'Brazil': 'South America',
    'Chile': 'South America',
    'Colombia': 'South America',
    'Ecuador': 'South America',
    'Falkland Islands': 'South America',
    'French Guiana': 'South America',
    'Guyana': 'South America',
    'Paraguay': 'South America',
    'Peru': 'South America',
    'Suriname': 'South America',
    'Uruguay': 'South America',
    'Venezuela': 'South America',
    'Central America, nfd': 'Central America',
    'Belize': 'Central America',
    'Costa Rica': 'Central America',
    'El Salvador': 'Central America',
    'Guatemala': 'Central America',
    'Honduras': 'Central America',
    'Mexico': 'Central America',
    'Nicaragua': 'Central America',
    'Panama': 'Central America',
    'Caribbean, nfd': 'Caribbean',
    'Anguilla': 'Caribbean',
    'Antigua and Barbuda': 'Caribbean',
    'Aruba': 'Caribbean',
    'Bahamas': 'Caribbean',
    'Barbados': 'Caribbean',
    'Cayman Islands': 'Caribbean',
    'Cuba': 'Caribbean',
    'Dominica': 'Caribbean',
    'Dominican Republic': 'Caribbean',
    'Grenada': 'Caribbean',
    'Guadeloupe': 'Caribbean',
    'Haiti': 'Caribbean',
    'Jamaica': 'Caribbean',
    'Martinique': 'Caribbean',
    'Montserrat': 'Caribbean',
    'Puerto Rico': 'Caribbean',
    'St Kitts and Nevis': 'Caribbean',
    'St Lucia': 'Caribbean',
    'St Vincent and the Grenadines': 'Caribbean',
    'Trinidad and Tobago': 'Caribbean',
    'Turks and Caicos Islands': 'Caribbean',
    'Virgin Islands, British': 'Caribbean',
    'Virgin Islands, United States': 'Caribbean',
    'St Martin (French part)': 'Caribbean',
    'Bonaire, Sint Eustatius and Saba': 'Caribbean',
    'Curacao': 'Caribbean',
    'Sint Maarten (Dutch part)': 'Caribbean',

    # Sub-Saharan Africa
    'Sub-Saharan Africa, nfd': 'Sub-Saharan Africa (nfd)',
    'Central and West Africa, nfd': 'Central and West Africa',
    'Benin': 'Central and West Africa',
    'Burkina Faso': 'Central and West Africa',
    'Cameroon': 'Central and West Africa',
    'Cabo Verde': 'Central and West Africa',
    'Central African Republic': 'Central and West Africa',
    'Chad': 'Central and West Africa',
    'Congo, Republic of': 'Central and West Africa',
    'Congo, Democratic Republic of': 'Central and West Africa',
    "Cote d'Ivoire": 'Central and West Africa',
    'Equatorial Guinea': 'Central and West Africa',
    'Gabon': 'Central and West Africa',
    'Gambia': 'Central and West Africa',
    'Ghana': 'Central and West Africa',
    'Guinea': 'Central and West Africa',
    'Guinea-Bissau': 'Central and West Africa',
    'Liberia': 'Central and West Africa',
    'Mali': 'Central and West Africa',
    'Mauritania': 'Central and West Africa',
    'Niger': 'Central and West Africa',
    'Nigeria': 'Central and West Africa',
    'Sao Tome and Principe': 'Central and West Africa',
    'Senegal': 'Central and West Africa',
    'Sierra Leone': 'Central and West Africa',
    'Togo': 'Central and West Africa',
    'Southern and East Africa, nfd': 'Southern and East Africa',
    'Angola': 'Southern and East Africa',
    'Botswana': 'Southern and East Africa',
    'Burundi': 'Southern and East Africa',
    'Comoros': 'Southern and East Africa',
    'Djibouti': 'Southern and East Africa',
    'Eritrea': 'Southern and East Africa',
    'Ethiopia': 'Southern and East Africa',
    'Kenya': 'Southern and East Africa',
    'Lesotho': 'Southern and East Africa',
    'Madagascar': 'Southern and East Africa',
    'Malawi': 'Southern and East Africa',
    'Mauritius': 'Southern and East Africa',
    'Mayotte': 'Southern and East Africa',
    'Mozambique': 'Southern and East Africa',
    'Namibia': 'Southern and East Africa',
    'Reunion': 'Southern and East Africa',
    'Rwanda': 'Southern and East Africa',
    'St Helena': 'Southern and East Africa',
    'Seychelles': 'Southern and East Africa',
    'Somalia': 'Southern and East Africa',
    'South Africa': 'Southern and East Africa',
    'Eswatini': 'Southern and East Africa',
    'Tanzania': 'Southern and East Africa',
    'Uganda': 'Southern and East Africa',
    'Zambia': 'Southern and East Africa',
    'Zimbabwe': 'Southern and East Africa'
}

In [127]:
##region
country_to_region = {
    # 1. Oceania and Antarctica
    'Oceania and Antarctica, nfd': 'Oceania and Antarctica',
    'New Zealand': 'Oceania and Antarctica',
    'Melanesia, nfd': 'Oceania and Antarctica',
    'New Caledonia': 'Oceania and Antarctica',
    'Papua New Guinea': 'Oceania and Antarctica',
    'Solomon Islands': 'Oceania and Antarctica',
    'Vanuatu': 'Oceania and Antarctica',
    'Micronesia, nfd': 'Oceania and Antarctica',
    'Guam': 'Oceania and Antarctica',
    'Kiribati': 'Oceania and Antarctica',
    'Marshall Islands': 'Oceania and Antarctica',
    'Micronesia, Federated States of': 'Oceania and Antarctica',
    'Nauru': 'Oceania and Antarctica',
    'Northern Mariana Islands': 'Oceania and Antarctica',
    'Palau': 'Oceania and Antarctica',
    'Cook Islands': 'Oceania and Antarctica',
    'Fiji': 'Oceania and Antarctica',
    'French Polynesia': 'Oceania and Antarctica',
    'Niue': 'Oceania and Antarctica',
    'Samoa': 'Oceania and Antarctica',
    'Samoa, American': 'Oceania and Antarctica',
    'Tokelau': 'Oceania and Antarctica',
    'Tonga': 'Oceania and Antarctica',
    'Tuvalu': 'Oceania and Antarctica',
    'Wallis and Futuna': 'Oceania and Antarctica',
    'Pitcairn Islands': 'Oceania and Antarctica',
    'Polynesia (excludes Hawaii), nec': 'Oceania and Antarctica',
    'Antarctica, nfd': 'Oceania and Antarctica',
    'Chilean Antarctic Territory': 'Oceania and Antarctica',

    # 2. North-West Europe
    'United Kingdom, Channel Islands and Isle of Ma': 'North-West Europe',
    'England': 'North-West Europe',
    'Isle of Man': 'North-West Europe',
    'Northern Ireland': 'North-West Europe',
    'Scotland': 'North-West Europe',
    'Wales': 'North-West Europe',
    'Guernsey': 'North-West Europe',
    'Jersey': 'North-West Europe',
    'Ireland': 'North-West Europe',
    'Western Europe, nfd': 'North-West Europe',
    'Austria': 'North-West Europe',
    'Belgium': 'North-West Europe',
    'France': 'North-West Europe',
    'Germany': 'North-West Europe',
    'Liechtenstein': 'North-West Europe',
    'Luxembourg': 'North-West Europe',
    'Monaco': 'North-West Europe',
    'Netherlands': 'North-West Europe',
    'Switzerland': 'North-West Europe',
    'Northern Europe, nfd': 'North-West Europe',
    'Denmark': 'North-West Europe',
    'Faroe Islands': 'North-West Europe',
    'Finland': 'North-West Europe',
    'Greenland': 'North-West Europe',
    'Iceland': 'North-West Europe',
    'Norway': 'North-West Europe',
    'Sweden': 'North-West Europe',
    'Aland Islands': 'North-West Europe',

    # 3. Southern and Eastern Europe
    'Southern and Eastern Europe, nfd': 'Southern and Eastern Europe',
    'Andorra': 'Southern and Eastern Europe',
    'Gibraltar': 'Southern and Eastern Europe',
    'Holy See': 'Southern and Eastern Europe',
    'Italy': 'Southern and Eastern Europe',
    'Malta': 'Southern and Eastern Europe',
    'Portugal': 'Southern and Eastern Europe',
    'San Marino': 'Southern and Eastern Europe',
    'Spain': 'Southern and Eastern Europe',
    'South Eastern Europe, nfd': 'Southern and Eastern Europe',
    'Albania': 'Southern and Eastern Europe',
    'Bosnia and Herzegovina': 'Southern and Eastern Europe',
    'Bulgaria': 'Southern and Eastern Europe',
    'Croatia': 'Southern and Eastern Europe',
    'Cyprus': 'Southern and Eastern Europe',
    'North Macedonia': 'Southern and Eastern Europe',
    'Greece': 'Southern and Eastern Europe',
    'Moldova': 'Southern and Eastern Europe',
    'Romania': 'Southern and Eastern Europe',
    'Slovenia': 'Southern and Eastern Europe',
    'Montenegro': 'Southern and Eastern Europe',
    'Serbia': 'Southern and Eastern Europe',
    'Kosovo': 'Southern and Eastern Europe',
    'Eastern Europe, nfd': 'Southern and Eastern Europe',
    'Belarus': 'Southern and Eastern Europe',
    'Czechia': 'Southern and Eastern Europe',
    'Estonia': 'Southern and Eastern Europe',
    'Hungary': 'Southern and Eastern Europe',
    'Latvia': 'Southern and Eastern Europe',
    'Lithuania': 'Southern and Eastern Europe',
    'Poland': 'Southern and Eastern Europe',
    'Russian Federation': 'Southern and Eastern Europe',
    'Slovakia': 'Southern and Eastern Europe',
    'Ukraine': 'Southern and Eastern Europe',

    # 4. North Africa and the Middle East
    'North Africa and the Middle East, nfd': 'North Africa and the Middle East',
    'North Africa, nfd': 'North Africa and the Middle East',
    'Algeria': 'North Africa and the Middle East',
    'Egypt': 'North Africa and the Middle East',
    'Libya': 'North Africa and the Middle East',
    'Morocco': 'North Africa and the Middle East',
    'Sudan': 'North Africa and the Middle East',
    'Tunisia': 'North Africa and the Middle East',
    'Western Sahara': 'North Africa and the Middle East',
    'South Sudan': 'North Africa and the Middle East',
    'Middle East, nfd': 'North Africa and the Middle East',
    'Bahrain': 'North Africa and the Middle East',
    'Gaza Strip and West Bank': 'North Africa and the Middle East',
    'Iran': 'North Africa and the Middle East',
    'Iraq': 'North Africa and the Middle East',
    'Israel': 'North Africa and the Middle East',
    'Jordan': 'North Africa and the Middle East',
    'Kuwait': 'North Africa and the Middle East',
    'Lebanon': 'North Africa and the Middle East',
    'Oman': 'North Africa and the Middle East',
    'Qatar': 'North Africa and the Middle East',
    'Saudi Arabia': 'North Africa and the Middle East',
    'Syria': 'North Africa and the Middle East',
    'Turkey': 'North Africa and the Middle East',
    'United Arab Emirates': 'North Africa and the Middle East',
    'Yemen': 'North Africa and the Middle East',

    # 5. South-East Asia
    'South-East Asia, nfd': 'South-East Asia',
    'Mainland South-East Asia, nfd': 'South-East Asia',
    'Myanmar': 'South-East Asia',
    'Cambodia': 'South-East Asia',
    'Laos': 'South-East Asia',
    'Thailand': 'South-East Asia',
    'Vietnam': 'South-East Asia',
    'Maritime South-East Asia, nfd': 'South-East Asia',
    'Brunei Darussalam': 'South-East Asia',
    'Indonesia': 'South-East Asia',
    'Malaysia': 'South-East Asia',
    'Philippines': 'South-East Asia',
    'Singapore': 'South-East Asia',
    'Timor-Leste': 'South-East Asia',

    # 6. North-East Asia
    'Chinese Asia (includes Mongolia), nfd': 'North-East Asia',
    'China (excludes SARs and Taiwan)': 'North-East Asia',
    'Hong Kong (SAR of China)': 'North-East Asia',
    'Macau (SAR of China)': 'North-East Asia',
    'Mongolia': 'North-East Asia',
    'Taiwan': 'North-East Asia',
    'Japan': 'North-East Asia',
    "Korea, Democratic People's Republic of (North)": 'North-East Asia',
    'Korea, Republic of (South)': 'North-East Asia',

    # 7. Southern and Central Asia
    'Southern and Central Asia, nfd': 'Southern and Central Asia',
    'Southern Asia, nfd': 'Southern and Central Asia',
    'Bangladesh': 'Southern and Central Asia',
    'Bhutan': 'Southern and Central Asia',
    'India': 'Southern and Central Asia',
    'Maldives': 'Southern and Central Asia',
    'Nepal': 'Southern and Central Asia',
    'Pakistan': 'Southern and Central Asia',
    'Sri Lanka': 'Southern and Central Asia',
    'Central Asia, nfd': 'Southern and Central Asia',
    'Afghanistan': 'Southern and Central Asia',
    'Armenia': 'Southern and Central Asia',
    'Azerbaijan': 'Southern and Central Asia',
    'Georgia': 'Southern and Central Asia',
    'Kazakhstan': 'Southern and Central Asia',
    'Kyrgyzstan': 'Southern and Central Asia',
    'Tajikistan': 'Southern and Central Asia',
    'Turkmenistan': 'Southern and Central Asia',
    'Uzbekistan': 'Southern and Central Asia',

    # 8. Americas
    'Americas, nfd': 'Americas',
    'Northern America, nfd': 'Americas',
    'Bermuda': 'Americas',
    'Canada': 'Americas',
    'United States of America': 'Americas',
    'South America, nfd': 'Americas',
    'Argentina': 'Americas',
    'Bolivia': 'Americas',
    'Brazil': 'Americas',
    'Chile': 'Americas',
    'Colombia': 'Americas',
    'Ecuador': 'Americas',
    'Falkland Islands': 'Americas',
    'French Guiana': 'Americas',
    'Guyana': 'Americas',
    'Paraguay': 'Americas',
    'Peru': 'Americas',
    'Suriname': 'Americas',
    'Uruguay': 'Americas',
    'Venezuela': 'Americas',
    'Central America, nfd': 'Americas',
    'Belize': 'Americas',
    'Costa Rica': 'Americas',
    'El Salvador': 'Americas',
    'Guatemala': 'Americas',
    'Honduras': 'Americas',
    'Mexico': 'Americas',
    'Nicaragua': 'Americas',
    'Panama': 'Americas',
    'Caribbean, nfd': 'Americas',
    'Anguilla': 'Americas',
    'Antigua and Barbuda': 'Americas',
    'Aruba': 'Americas',
    'Bahamas': 'Americas',
    'Barbados': 'Americas',
    'Cayman Islands': 'Americas',
    'Cuba': 'Americas',
    'Dominica': 'Americas',
    'Dominican Republic': 'Americas',
    'Grenada': 'Americas',
    'Guadeloupe': 'Americas',
    'Haiti': 'Americas',
    'Jamaica': 'Americas',
    'Martinique': 'Americas',
    'Montserrat': 'Americas',
    'Puerto Rico': 'Americas',
    'St Kitts and Nevis': 'Americas',
    'St Lucia': 'Americas',
    'St Vincent and the Grenadines': 'Americas',
    'Trinidad and Tobago': 'Americas',
    'Turks and Caicos Islands': 'Americas',
    'Virgin Islands, British': 'Americas',
    'Virgin Islands, United States': 'Americas',
    'St Martin (French part)': 'Americas',
    'Bonaire, Sint Eustatius and Saba': 'Americas',
    'Curacao': 'Americas',
    'Sint Maarten (Dutch part)': 'Americas',

    # 9. Sub-Saharan Africa
    'Sub-Saharan Africa, nfd': 'Sub-Saharan Africa',
    'Central and West Africa, nfd': 'Sub-Saharan Africa',
    'Benin': 'Sub-Saharan Africa',
    'Burkina Faso': 'Sub-Saharan Africa',
    'Cameroon': 'Sub-Saharan Africa',
    'Cabo Verde': 'Sub-Saharan Africa',
    'Central African Republic': 'Sub-Saharan Africa',
    'Chad': 'Sub-Saharan Africa',
    'Congo, Republic of': 'Sub-Saharan Africa',
    'Congo, Democratic Republic of': 'Sub-Saharan Africa',
    "Cote d'Ivoire": 'Sub-Saharan Africa',
    'Equatorial Guinea': 'Sub-Saharan Africa',
    'Gabon': 'Sub-Saharan Africa',
    'Gambia': 'Sub-Saharan Africa',
    'Ghana': 'Sub-Saharan Africa',
    'Guinea': 'Sub-Saharan Africa',
    'Guinea-Bissau': 'Sub-Saharan Africa',
    'Liberia': 'Sub-Saharan Africa',
    'Mali': 'Sub-Saharan Africa',
    'Mauritania': 'Sub-Saharan Africa',
    'Niger': 'Sub-Saharan Africa',
    'Nigeria': 'Sub-Saharan Africa',
    'Sao Tome and Principe': 'Sub-Saharan Africa',
    'Senegal': 'Sub-Saharan Africa',
    'Sierra Leone': 'Sub-Saharan Africa',
    'Togo': 'Sub-Saharan Africa',
    'Southern and East Africa, nfd': 'Sub-Saharan Africa',
    'Angola': 'Sub-Saharan Africa',
    'Botswana': 'Sub-Saharan Africa',
    'Burundi': 'Sub-Saharan Africa',
    'Comoros': 'Sub-Saharan Africa',
    'Djibouti': 'Sub-Saharan Africa',
    'Eritrea': 'Sub-Saharan Africa',
    'Ethiopia': 'Sub-Saharan Africa',
    'Kenya': 'Sub-Saharan Africa',
    'Lesotho': 'Sub-Saharan Africa',
    'Madagascar': 'Sub-Saharan Africa',
    'Malawi': 'Sub-Saharan Africa',
    'Mauritius': 'Sub-Saharan Africa',
    'Mayotte': 'Sub-Saharan Africa',
    'Mozambique': 'Sub-Saharan Africa',
    'Namibia': 'Sub-Saharan Africa',
    'Reunion': 'Sub-Saharan Africa',
    'Rwanda': 'Sub-Saharan Africa',
    'St Helena': 'Sub-Saharan Africa',
    'Seychelles': 'Sub-Saharan Africa',
    'Somalia': 'Sub-Saharan Africa',
    'South Africa': 'Sub-Saharan Africa',
    'Eswatini': 'Sub-Saharan Africa',
    'Tanzania': 'Sub-Saharan Africa',
    'Uganda': 'Sub-Saharan Africa',
    'Zambia': 'Sub-Saharan Africa',
    'Zimbabwe': 'Sub-Saharan Africa'
}

In [129]:
combined_df["Region_class"]=combined_df["Country of birth of person"].map(country_to_region)

In [130]:
combined_df["Subregion_class"]=combined_df["Country of birth of person"].map(country_to_subregion)

In [131]:
combined_df.fillna("Not specified",inplace=True)

In [132]:
combined_df.isnull().sum()

Country of birth of person                                 0
Years spent in Australia                                   0
Age group                                                  0
Sex                                                        0
Long-term health condition (LTHC)                          0
Number of people reporting LTHC(s)                         0
Population                                                 0
Age-specific percentage of population reporting LTHC(s)    0
Language used at home                                      0
Proficiency in spoken English                              0
Region_class                                               0
Subregion_class                                            0
dtype: int64

In [134]:
combined_df.to_csv("Cleaned_data_final.csv")